# U-Net z próbkowaniem inspirowanym portem Budy

Sprawdzamy wpływ częstszego losowania przekrojów zawierających adnotację przy zachowaniu dotychczasowego U-Neta, preprocessingu i funkcji straty. Ten etap obejmuje trzy przebiegi z seedami 42, 43 i 44 na stałym podziale pacjentów.

Trening ma 2714 przykładów i 77 pacjentów, a walidacja 615 przykładów i 16 pacjentów. W każdym przebiegu używamy świeżej inicjalizacji, Adama z `lr=3e-4`, paczek po 4 przekroje, BCE + soft Dice i 40 epok. Checkpoint wybieramy według najwyższego walidacyjnego `dice_annotated`, przy progu `> 0.5`.

Domyślne `run_training = False` służy do przeglądania przygotowania danych i zapisanych wyników. Nowy przebieg wymaga własnego `run_name` i włączenia treningu. Ocenę po pacjentach wykonujemy w notebooku 08. Zbiór testowy ma zaplanowaną rolę końcowej oceny.

## 1. Biblioteki, ścieżki i konfiguracja

Importujemy U-Neta z `src/models.py` i wyznaczamy katalog projektu. Seed steruje inicjalizacją wag oraz generatorem samplera. Podział pacjentów pochodzi z istniejącego `splits.csv`.

Konfiguracja opisuje wybrany przebieg. Pętla treningowa sprawdza, czy pliki checkpointu i historii już istnieją, zanim rozpocznie obliczenia.

In [1]:
from pathlib import Path
import sys

import torch
import pandas as pd
import tifffile
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import WeightedRandomSampler

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.models import UNet

In [ ]:
run_training = False
run_name = "unet_buda_sampling_seed44"
seed = 44
learning_rate = 3e-4
batch_size = 4
num_epochs = 40

## 2. Metadane i stały podział pacjentów

Łączymy tabele po `patient_id` i tworzymy `train_df` oraz `val_df`. Wszystkie dotychczasowe pary, również z flagą `anomaly_candidate`, należą do swoich zapisanych podziałów. Kolumna `foreground_pixels` zawiera powierzchnię adnotacji i posłuży do przygotowania wag.

In [2]:
pairs_df = pd.read_csv(project_root / "data" / "metadata" / "pairs.csv")
splits_df = pd.read_csv(project_root / "data" / "splits" / "splits.csv")
pairs_with_split = pairs_df.merge(
    splits_df, on="patient_id", how="left", validate="many_to_one"
)

val_df = pairs_with_split.loc[pairs_with_split["split"] == "val"].copy()

train_df = pairs_with_split.loc[pairs_with_split["split"] == "train"].copy()
print(f"Train: {len(train_df)} przekrojów, val: {len(val_df)} przekrojów.")

Train: 2714 przekrojów, val: 615 przekrojów.


## 3. Mechanizm próbkowania i przykład jednego pacjenta

Dla przekroju o powierzchni adnotacji `a_i` i pacjenta z `n` przekrojami stosujemy prawdopodobieństwo warunkowe

`p_i = (a_i + 0.1 * mean(a)) / (1.1 * sum(a))`.

Dodatek 10% średniej powierzchni dotyczy każdego przekroju pacjenta, także pustej maski. Po normalizacji wagi przekrojów jednego pacjenta sumują się do 1. Gdy te wagi przekazujemy wspólnie do samplera, każdy z 77 pacjentów otrzymuje taki sam łączny udział, a większe adnotacje zwiększają udział przekroju wewnątrz pacjenta.

Port Budy losuje najpierw pacjenta równomiernie, potem jego przekrój według takich wag. `WeightedRandomSampler` ze zwracaniem realizuje ten sam rozkład łączny na naszej tabeli. To adaptacja reguły próbkowania na naszym podziale i oryginalnych maskach TIFF. W porcie wagi powstają po jego preprocessingu. Opis z publikacji obejmuje powielanie oznaczonych przekrojów z obrotem i skalowaniem.

Źródła metody są w [dataset.py portu PyTorch](https://github.com/mateuszbuda/brain-segmentation-pytorch/blob/master/dataset.py#L74-L116) oraz [sekcji 3.1.2 publikacji](https://arxiv.org/html/1906.03720#S3.SS1.SSS2).

In [3]:
train_df[["patient_id", "slice_id", "foreground_pixels"]].head(23)

,patient_id,slice_id,foreground_pixels
0,TCGA_CS_4941_19960909,1,0
1,TCGA_CS_4941_19960909,2,0
2,TCGA_CS_4941_19960909,3,0
3,TCGA_CS_4941_19960909,4,0
4,TCGA_CS_4941_19960909,5,0
5,TCGA_CS_4941_19960909,6,0
6,TCGA_CS_4941_19960909,7,0
7,TCGA_CS_4941_19960909,8,0
8,TCGA_CS_4941_19960909,9,0
9,TCGA_CS_4941_19960909,10,0


In [4]:
patient_id = train_df.iloc[0]["patient_id"]
patient = train_df.loc[train_df["patient_id"] == patient_id,["slice_id", "foreground_pixels"]].copy()
area = patient["foreground_pixels"]
bonus = 0.1 * area.sum() / len(patient)
weights = area + bonus
patient["sampling_pct"] = 100 * weights / weights.sum()

patient

,slice_id,foreground_pixels,sampling_pct
0,1,0,0.395257
1,2,0,0.395257
2,3,0,0.395257
3,4,0,0.395257
4,5,0,0.395257
5,6,0,0.395257
6,7,0,0.395257
7,8,0,0.395257
8,9,0,0.395257
9,10,0,0.395257


## 4. Wagi treningowe i kontrola odczytu paczki

Liczymy średnią i sumę osobno dla każdego pacjenta za pomocą `groupby` i `transform`. W badanym train każdy pacjent ma co najmniej jeden oznaczony przekrój. Kontrola sum wag powinna pokazać 77 wartości równych 1 z dokładnością do obliczeń numerycznych.

Przy zwykłym przejściu po train około 33,97% przekrojów ma adnotację. Dla zastosowanego rozkładu oczekiwany udział wynosi około 94,07%. Liczba losowań w epoce pozostaje równa 2714, co daje 679 aktualizacji przy batch size 4 i 27 160 aktualizacji w 40 epokach. Powtórzone indeksy są dopuszczalne.

`MRIDataset` odczytuje TIFF-y na żądanie. Obraz ma `float32`, kształt `(3, 256, 256)` i zakres `[0, 1]`. Maska ma `float32`, kształt `(1, 256, 256)` i wartości 0/1. `copy=True` przekazuje samplerowi zapisywalną kopię tablicy wag. Osobny generator samplera treningowego zostaje utworzony na początku właściwego przebiegu.

In [5]:
patient_mean = train_df.groupby("patient_id")["foreground_pixels"].transform("mean")
bonus = patient_mean * 0.1

train_df["sampling_weight"] = train_df["foreground_pixels"] + bonus
train_df["sampling_prob"] = (
    train_df["sampling_weight"] 
    / train_df.groupby("patient_id")["sampling_weight"].transform("sum")
)
train_df.groupby("patient_id")["sampling_prob"].sum().describe()

count    7.700000e+01
mean     1.000000e+00
std      4.932295e-17
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: sampling_prob, dtype: float64

In [6]:
def preprocess_pair(image, mask):
    image_tensor = torch.from_numpy(image).permute(2, 0, 1).float() / 255
    mask_tensor = torch.from_numpy(mask > 0).float().unsqueeze(0)

    return (image_tensor, mask_tensor)

class MRIDataset(Dataset):
    def __init__(self, pairs_df, project_root, transform=None):
        self.pairs_df = pairs_df
        self.project_root = project_root
        self.transform = transform

    def __len__(self):
        return self.pairs_df.shape[0]

    def __getitem__(self, index):
        row = self.pairs_df.iloc[index]
        image_path = self.project_root / row["image_path"]
        mask_path = self.project_root / row["mask_path"]
        image, mask = preprocess_pair(tifffile.imread(image_path), tifffile.imread(mask_path))

        if self.transform is not None:
            return self.transform(image, mask)
        return image, mask

In [7]:
train_dataset = MRIDataset(train_df, project_root)

train_sampler = WeightedRandomSampler(
    weights=train_df["sampling_prob"].to_numpy(copy=True),
    num_samples=len(train_dataset),
    replacement=True,
    generator=torch.Generator().manual_seed(seed),
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=train_sampler)

images, masks = next(iter(train_loader))
print(len(train_loader), images.shape, masks.shape)

679 torch.Size([4, 3, 256, 256]) torch.Size([4, 1, 256, 256])


## 5. Funkcja straty, trening i walidacja

BCE otrzymuje surowe logity. Soft Dice liczymy po sigmoidzie osobno dla każdego przekroju i uśredniamy w paczce, z `eps=1e-5`. Obie składowe straty mają wagę 1.

Trening wykorzystuje sampler, a walidacja przechodzi po wszystkich przekrojach w stałej kolejności. Raportujemy Dice wszystkich przekrojów, przekrojów z adnotacją i pustych masek. Dwie puste maski otrzymują 1, a niepusta predykcja przy pustym wzorcu otrzymuje 0.

In [8]:
def soft_dice_loss(logits, targets, eps=1e-5):
    p = torch.sigmoid(logits)

    num = 2*(p*targets).sum(dim=(1, 2, 3)) + eps
    den = p.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3)) + eps

    return 1 - (num/den).mean()

bce_criterion = torch.nn.BCEWithLogitsLoss()

def bce_dice_loss(logits, targets):
    bce = bce_criterion(logits, targets)
    soft_dice = soft_dice_loss(logits, targets)

    return bce + soft_dice

In [9]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    n_samples = 0

    for images, masks in loader:

        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, masks)

        loss.backward()
        optimizer.step()

        batch_size = images.shape[0]
        total_loss += loss.item() * batch_size
        n_samples += batch_size

    return total_loss / n_samples

In [10]:
def dice_per_sample(predictions, targets):
    targets_logic = targets > 0

    targets_sum = targets_logic.sum(dim=(1, 2, 3))
    predictions_sum = predictions.sum(dim=(1, 2, 3))
    intersection = (targets_logic & predictions).sum(dim=(1, 2, 3))

    denominator = targets_sum + predictions_sum

    return torch.where(
        denominator == 0,
        1.0,
        2 * intersection / denominator.clamp_min(1)
    )


def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    n_samples = 0
    total_dice = 0.0
    total_dice_annotated = 0.0
    total_dice_empty = 0.0
    n_annotated = 0
    n_empty = 0

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)

            logits = model(images)
            loss = criterion(logits, masks)

            predictions = torch.sigmoid(logits) > 0.5
            dice_scores = dice_per_sample(predictions, masks)
            has_annotation = masks.sum(dim=(1, 2, 3)) > 0

            batch_size = images.shape[0]
            total_loss += loss.item() * batch_size

            n_samples += batch_size
            n_annotated += has_annotation.sum().item()
            n_empty += (~has_annotation).sum().item()

            total_dice += dice_scores.sum().item()
            total_dice_annotated += dice_scores[has_annotation].sum().item()
            total_dice_empty += dice_scores[~has_annotation].sum().item()

    assert n_annotated + n_empty == n_samples

    return {
        "loss": total_loss / n_samples,
        "dice": total_dice / n_samples,
        "dice_annotated": total_dice_annotated / n_annotated,
        "dice_empty": total_dice_empty / n_empty
    }

## 6. Uruchomienie przebiegu i zapis checkpointu

Każdy przebieg tworzy świeży model, optymalizator i generator losowania. Checkpoint aktualizujemy przy wzroście `dice_annotated`. Zawiera wagi, stan optymalizatora, seed, epokę i historię do chwili zapisu. Pełny CSV historii powstaje po ostatniej epoce.

Zachowany wydruk dokumentuje wykonany trening. Tabela poniżej odczytuje wyniki wszystkich zakończonych powtórzeń z lokalnych historii i ocen z notebooka 08.

In [12]:
if run_training:
    best_checkpoint_path = (
        project_root / "outputs" / "checkpoints" / f"{run_name}_best.pt"
    )
    history_path = project_root / "outputs" / f"{run_name}_history.csv"
    for path in (best_checkpoint_path, history_path):
        if path.exists():
            raise FileExistsError(f"Wybierz inną nazwę przebiegu. Plik już istnieje: {path}")
    best_checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

    torch.manual_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = UNet().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    train_generator = torch.Generator().manual_seed(seed)

    train_dataset = MRIDataset(train_df, project_root)
    val_dataset = MRIDataset(val_df, project_root)

    train_sampler = WeightedRandomSampler(
        weights=train_df["sampling_prob"].to_numpy(copy=True),
        num_samples=len(train_dataset),
        replacement=True,
        generator=train_generator,
    )
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, sampler=train_sampler
    )
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    history = []
    best_val_dice = -float("inf")

    for epoch in range(1, num_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, bce_dice_loss, optimizer, device)
        val_metrics = validate_one_epoch(model, val_loader, bce_dice_loss, device)
        history.append({"epoch": epoch, "train_loss": train_loss, **val_metrics})

        current_dice = val_metrics["dice_annotated"]
        print(f"Epoka {epoch}/{num_epochs}, train loss = {train_loss:.4f}")
        print("Walidacja", val_metrics)

        if current_dice > best_val_dice:
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "history": history,
                    "seed": seed,
                },
                best_checkpoint_path,
            )
            best_val_dice = current_dice
            print(f"Zapisano najlepszy model z epoki {epoch}, Dice = {current_dice:.4f}")

    pd.DataFrame(history).to_csv(history_path, index=False)
    print(f"Zapisano {len(history)} epok w {history_path.name}")

Epoka 1/40, train loss = 0.6560
Walidacja {'loss': 0.9890838245066201, 'dice': 0.5845076348602287, 'dice_annotated': 0.2901364230566611, 'dice_empty': 0.7678100263852242}
Zapisano najlepszy model z epoki 1, Dice = 0.2901
Epoka 2/40, train loss = 0.5313
Walidacja {'loss': 0.9691716985973885, 'dice': 0.49075270658586084, 'dice_annotated': 0.3212411634750285, 'dice_empty': 0.5963060686015831}
Zapisano najlepszy model z epoki 2, Dice = 0.3212
Epoka 3/40, train loss = 0.5109
Walidacja {'loss': 0.9440400887310989, 'dice': 0.4755030103512411, 'dice_annotated': 0.3408235222006501, 'dice_empty': 0.5593667546174143}
Zapisano najlepszy model z epoki 3, Dice = 0.3408
Epoka 4/40, train loss = 0.4657
Walidacja {'loss': 0.9269500693170035, 'dice': 0.3813230064099397, 'dice_annotated': 0.4598035965809378, 'dice_empty': 0.3324538258575198}
Zapisano najlepszy model z epoki 4, Dice = 0.4598
Epoka 5/40, train loss = 0.3264
Walidacja {'loss': 0.875441899483766, 'dice': 0.42050836583887174, 'dice_annotated'

## 7. Porównanie zapisanych powtórzeń

Każdy wiersz dotyczy checkpointu wybranego według `dice_annotated`. Epokę odczytujemy z pełnej historii, a pięć metryk z ponownej oceny checkpointu w notebooku 08. Dzięki temu metryki po przekrojach i po pacjentach opisują te same wagi. Drobne różnice ostatnich cyfr względem historii mogą wynikać z ponownego obliczenia na CPU.

In [ ]:
summary_rows = []
missing_results = []

for run_seed in (42, 43, 44):
    prefix = f"unet_buda_sampling_seed{run_seed}"
    history_path = project_root / "outputs" / f"{prefix}_history.csv"
    metrics_path = project_root / "outputs" / f"{prefix}_val_metrics.csv"
    if not history_path.is_file() or not metrics_path.is_file():
        missing_results.append(prefix)
        continue

    run_history = pd.read_csv(history_path)
    best = run_history.loc[run_history["dice_annotated"].idxmax()]
    saved_metrics = pd.read_csv(metrics_path, index_col=0).iloc[:, 0]
    summary_rows.append({
        "seed": run_seed,
        "epoki": len(run_history),
        "wybrana epoka": int(best["epoch"]),
        "Dice ogólny": saved_metrics["Dice po wszystkich przekrojach"],
        "Dice oznaczonych": saved_metrics["Dice przekrojów z adnotacją"],
        "Dice pustych": saved_metrics["Dice przekrojów z pustą maską"],
        "Dice 3D": saved_metrics["Dice 3D po pacjentach"],
        "Dice 3D po filtrze": saved_metrics["Dice 3D po pacjentach po filtrowaniu"],
    })

if missing_results:
    print("Brak lokalnych wyników dla:", ", ".join(missing_results))

sampling_summary = pd.DataFrame(summary_rows)
sampling_summary.round(4)

,seed,epoki,wybrana epoka,Dice ogólny,Dice oznaczonych,Dice pustych,Dice 3D,Dice 3D po filtrze
0,42,40,36,0.7505,0.6084,0.8391,0.6561,0.6651
1,43,40,27,0.5684,0.6380,0.5251,0.6445,0.6775
2,44,40,18,0.5488,0.7224,0.4406,0.6756,0.7589


## Wnioski

Dla seedów 42, 43 i 44 średni Dice oznaczonych przekrojów wyniósł 0,6563 ± 0,0592, wobec 0,4716 ± 0,0108 dla bazowego U-Neta. Próbkowanie poprawiło tę metrykę na każdym z trzech seedów.

Średni Dice 3D wzrósł z 0,4686 ± 0,0344 do 0,6587 ± 0,0157, a po największej składowej 3D z 0,5309 ± 0,0279 do 0,7005 ± 0,0510. Podane odchylenia są odchyleniami standardowymi z próby między trzema seedami. Każdy wynik 3D jest średnią po tych samych 16 pacjentach val.

Reguła zmienia zarówno udział pacjentów, jak i rozkład przekrojów wewnątrz pacjenta. Wnioski dotyczą całego zastosowanego sposobu próbkowania. Następny zakończony etap, opisany w notebooku 10, dokłada standaryzację kanałów. Zbiorcze zestawienie wszystkich przebiegów znajduje się na końcu notebooka 10.